Baseline Model 2: Random Forest

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score

In [ ]:
train_input = "new_data/mover_epic_final_train_features.csv"
test_input = "new_data/mover_epic_final_test_features.csv"

print("[Random Forest GridSearch] Ingesting synchronized contract datasets...")
if not os.path.exists(train_input) or not os.path.exists(test_input):
    raise FileNotFoundError("Missing essential upstream processed cohort files.")

train_df = pd.read_csv(train_input, encoding='utf-8')
test_df = pd.read_csv(test_input, encoding='utf-8')

# Force absolute sorted chronological sequence to prevent look-ahead bias
if 'IN_OR_DTTM' in train_df.columns:
    train_df['IN_OR_DTTM'] = pd.to_datetime(train_df['IN_OR_DTTM'])
    train_df = train_df.sort_values('IN_OR_DTTM', ascending=True).reset_index(drop=True)


In [ ]:
print("[Random Forest GridSearch] Structuring explicit contract whitelist...")

numeric_features = ['AGE', 'HEIGHT', 'WEIGHT', 'SCHEDULED_START_HOUR', 'SURGERY_DAY_OF_WEEK', 'SURGERY_MONTH',
                    'DOW_SIN', 'DOW_COS', 'MONTH_SIN', 'MONTH_COS', 'HOUR_SIN', 'HOUR_COS']
binary_features = ['SEX_CODE', 'ICU_ADMIN_FLAG_CODE', 'IS_MORNING_CASE_CODE', 'Is_Hypertension', 'Is_Diabetes', 'Is_Cardiac']
ordinal_features = ['ASA_RATING_C']

# Automatically extract sparse dynamic columns generated via OneHotEncoder
procedure_features = [c for c in train_df.columns if c.startswith("PRIMARY_PROCEDURE_NM_")]

# Enforce strict alphabetical dictionary order to shield against array position drift
feature_cols = sorted(numeric_features + binary_features + ordinal_features + procedure_features)

# Hard bi-directional subset assertions to filter any un-authorized operational metrics
leakage_columns = [
    'LOG_ID', 'MRN', 'CASE_ID', 'SEX', 'PRIMARY_PROCEDURE_NM', 'ASA_RATING', 'ICU_ADMIN_FLAG', 'IS_MORNING_CASE',
    'IN_OR_DTTM', 'OUT_OR_DTTM', 'AN_START_DATETIME', 'AN_STOP_DATETIME', 'SURGERY_DATE', 'BIRTH_DATE',
    'HOSP_ADMSN_TIME', 'HOSP_DISCH_TIME', 'ACTUAL_DURATION', 'TARGET_LOG', 'DISCH_DISP'
]

for split_name, cohort_df in [("Train", train_df), ("Test", test_df)]:
    active_cols = [col for col in cohort_df.columns if col not in leakage_columns and not col.startswith("INTERACT_")]
    missing = set(feature_cols) - set(active_cols)
    extra = set(active_cols) - set(feature_cols)
    assert len(missing) == 0, f"Defensive Intercept! Missing metrics inside {split_name} block: {missing}"
    assert len(extra) == 0, f"Defensive Intercept! Un-authorized extra features inside {split_name} block: {extra}"

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train_raw = train_df['ACTUAL_DURATION'].copy()
y_train_log = np.log1p(y_train_raw)
y_test_true = test_df['ACTUAL_DURATION'].copy()

In [ ]:
print("[Random Forest GridSearch] Instantiating isolated preprocessor flows...")

numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median'))])
binary_pipeline = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value=0))])
ordinal_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent'))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('bin', binary_pipeline, binary_features),
        ('ord', ordinal_pipeline, ordinal_features),
        ('proc', 'passthrough', procedure_features)
    ]
)

In [ ]:
rf_model = RandomForestRegressor(random_state=42, n_jobs=-1)

pipe = Pipeline([
    ("preprocess", preprocessor),
    ("model", rf_model)
])

print("[Random Forest GridSearch] Setting up TimeSeriesSplit CV grid optimization...")
tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [15, 20],
    "model__min_samples_leaf": [10],
    "model__max_features": ["sqrt"]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=2
)

print("[Random Forest GridSearch] Fitting automated hyperparameter matrix grids smoothly...")
grid.fit(X_train, y_train_log)

best_model = grid.best_estimator_
print(f"Optimization Complete! Best Parameter Configuration: {grid.best_params_}")
print(f" Validation CVlog-MAE (Best Score): {-grid.best_score_:.4f}")


In [ ]:
print("[Evaluation] Executing forward prospective inference over sequestered 2022 cohort...")

pred_log = best_model.predict(X_test)
y_test_log_true = np.log1p(y_test_true)

# Convert log manifold predictions back to physical minutes scale safely
pred_minutes = np.expm1(pred_log)

print("\n" + "="*75)
print("     FINAL PROSPECTIVE TEST RESULTS (2022 SEQUESTERED COHORT)")
print("="*75)
print("--- Absolute Clinical Minutes Scale (Downstream Tactical Reference) ---")
print(f"   MAE (Mean Absolute Error)          : {mean_absolute_error(y_test_true, pred_minutes):.2f} minutes")
print(f"   RMSE (Root Mean Squared Error)     : {np.sqrt(mean_squared_error(y_test_true, pred_minutes)):.2f} minutes")
print(f"   MedianAE (Median Absolute Error)   : {median_absolute_error(y_test_true, pred_minutes):.2f} minutes")
print(f"   R2 Score (Variance Explained)      : {r2_score(y_test_true, pred_minutes):.4f}")
print("-"*75)
print("--- Internal Log-Transformed Target Manifold (Structural Reference) ---")
print(f"   Log-MAE                            : {mean_absolute_error(y_test_log_true, pred_log):.4f}")
print(f"   Log-R2 Score                       : {r2_score(y_test_log_true, pred_log):.4f}")
print("="*75 + "\n")

In [ ]:
print("[Random Forest GridSearch] Logging perfectly-aligned feature importance indices...")

raw_feature_names = best_model.named_steps["preprocess"].get_feature_names_out()
cleaned_feature_names = [x.split("__")[-1] for x in raw_feature_names]

importance_df = pd.DataFrame({
    "Feature": cleaned_feature_names,
    "Importance": best_model.named_steps["model"].feature_importances_
})

importance_df = importance_df.sort_values("Importance", ascending=False).reset_index(drop=True)

print("Top 20 Features Audited:")
print(importance_df.head(20))

os.makedirs("new_data", exist_ok=True)
importance_df.to_csv("new_data/random_forest_feature_importance.csv", index=False, encoding='utf-8')
print("\nFeature attributions saved successfully: new_data/random_forest_feature_importance.csv")